In [1]:
from glob import glob
tif_list = glob('/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/her2st/test*')
tif_list.sort()
test_slides = list()
for tif in tif_list:
    tif_path = tif.split('_')[-1].split('.')[0]
    test_slides.append(tif_path)
test_slides

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

In [3]:
import joblib
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from hilearn.plot import corr_plot

cell_names = [cell[23:] for cell in list(pd.read_csv("/data1/r20user3/shared_project/Hist2Cell/data/her2st/A1/cell_ratio.csv").columns)[1:]]
for case in test_slides:
    os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case))
    
    save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/her2st/her2st_epoch100_lr1e-4_2hop_ensemble_onlycell_"+case+"_best_cell_all_abundance_average.pkl"
    pred_and_label = joblib.load(save_path)
    
    case_pred = list()
    case_label = list()
    
    for slide in pred_and_label:
        os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, slide))
        
        pred_and_label[slide]['cell_abundance_predictions'] = np.clip(pred_and_label[slide]['cell_abundance_predictions'], a_min=0, a_max=None)
        pred_ratio = pred_and_label[slide]['cell_abundance_predictions'] / pred_and_label[slide]['cell_abundance_predictions'].sum(axis=1, keepdims=True)
        real_ratio = pred_and_label[slide]['cell_abundance_labels'] / pred_and_label[slide]['cell_abundance_labels'].sum(axis=1, keepdims=True)
        
        os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, slide, "cell proportion"))
        for i in range(39):
            plt.figure(figsize=(4, 4))
            x = np.squeeze(pred_ratio[:,[i]])
            y = np.squeeze(real_ratio[:,[i]])

            corr_plot(x, y)
            plt.title(cell_names[i])
            plt.xlabel("Inferred cell proportion")
            plt.ylabel("Cell2location cell proportion")
            # plt.show()
            plt.savefig(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, slide, "cell proportion", cell_names[i]+".png"))
            plt.close()

            
        os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, slide, "cell abundance"))
        for i in range(39):
            plt.figure(figsize=(4, 4))
            x = np.squeeze(pred_and_label[slide]['cell_abundance_predictions'][:,[i]])
            y = np.squeeze(pred_and_label[slide]['cell_abundance_labels'][:,[i]])

            corr_plot(x, y)
            plt.title(cell_names[i])
            plt.xlabel("Inferred cell abundance")
            plt.ylabel("Cell2location cell abundance")
            # plt.show()
            plt.savefig(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, slide, "cell abundance", cell_names[i]+".png"))
            plt.close()
        
        case_pred.append(pred_and_label[slide]['cell_abundance_predictions'])
        case_label.append(pred_and_label[slide]['cell_abundance_labels'])
    
    case_pred = np.concatenate(case_pred)
    case_label = np.concatenate(case_label)
    pred_ratio = case_pred / case_pred.sum(axis=1, keepdims=True)
    real_ratio = case_label / case_label.sum(axis=1, keepdims=True)
    os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, "cell proportion"))
    for i in range(39):
        plt.figure(figsize=(4, 4))
        x = np.squeeze(pred_ratio[:,[i]])
        y = np.squeeze(real_ratio[:,[i]])

        corr_plot(x, y)
        plt.title(cell_names[i])
        plt.xlabel("Inferred cell proportion")
        plt.ylabel("Cell2location cell proportion")
        # plt.show()
        plt.savefig(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, "cell proportion", cell_names[i]+".png"))
        plt.close()

        
    os.mkdir(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, "cell abundance"))
    for i in range(39):
        plt.figure(figsize=(4, 4))
        x = np.squeeze(case_pred[:,[i]])
        y = np.squeeze(case_label[:,[i]])

        corr_plot(x, y)
        plt.title(cell_names[i])
        plt.xlabel("Inferred cell proportion")
        plt.ylabel("Cell2location cell proportion")
        # plt.show()
        plt.savefig(os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/analysis/pearson_analysis/pearson_corr_plot_each_case_her2st", case, "cell abundance", cell_names[i]+".png"))
        plt.close()